# Session 18 Lab: Multimodal AI
**Course:** Noob to AI Expert | **Track:** Expert

Analyze images with Claude, transcribe audio with Whisper, and build a multimodal pipeline.

In [ ]:
!pip install anthropic openai requests pillow -q
print('✅ Ready')

In [ ]:
import anthropic, base64, os, requests
from pathlib import Path

os.environ.setdefault('ANTHROPIC_API_KEY', 'your-key-here')
client = anthropic.Anthropic()

def analyze_image_url(url, question):
    resp = client.messages.create(
        model='claude-opus-4-7',
        max_tokens=500,
        messages=[{
            'role': 'user',
            'content': [
                {'type': 'image', 'source': {'type': 'url', 'url': url}},
                {'type': 'text', 'text': question}
            ]
        }]
    )
    return resp.content[0].text

def analyze_image_file(path, question):
    data = base64.standard_b64encode(Path(path).read_bytes()).decode()
    ext = Path(path).suffix.lower()
    mtype = {'.jpg': 'image/jpeg', '.jpeg': 'image/jpeg', '.png': 'image/png'}.get(ext, 'image/jpeg')
    resp = client.messages.create(
        model='claude-opus-4-7',
        max_tokens=500,
        messages=[{'role': 'user', 'content': [
            {'type': 'image', 'source': {'type': 'base64', 'media_type': mtype, 'data': data}},
            {'type': 'text', 'text': question}
        ]}]
    )
    return resp.content[0].text

# Test with a sample image
sample_url = 'https://upload.wikimedia.org/wikipedia/commons/thumb/d/d9/Collage_of_Nine_Dogs.jpg/600px-Collage_of_Nine_Dogs.jpg'
result = analyze_image_url(sample_url, 'Describe what you see. How many dogs are there? What breeds can you identify?')
print(result)

In [ ]:
# Build a simple multimodal document index from images
image_urls = [
    ('chart.png', 'https://upload.wikimedia.org/wikipedia/commons/thumb/1/1d/World_population_growth_rate_1950%E2%80%932050.svg/600px-World_population_growth_rate_1950%E2%80%932050.svg.png'),
]

index = []
for name, url in image_urls:
    print(f'Extracting content from {name}...')
    extracted = analyze_image_url(url, 'Extract ALL text, numbers, and data from this image. Include all labels, values, and captions.')
    index.append({'source': name, 'content': extracted})
    print(f'  Extracted {len(extracted)} chars')

print('\nIndex built. Querying...')
query = 'What trends does the data show?'
context = '\n\n'.join(f'[{d["source"]}]\n{d["content"]}' for d in index)

final = client.messages.create(
    model='claude-haiku-4-5-20251001', max_tokens=300,
    system='Answer only based on the provided document context.',
    messages=[{'role': 'user', 'content': f'Context:\n{context}\n\nQuestion: {query}'}]
).content[0].text
print(f'Answer: {final}')